# Five-method main comparison
PN, GPN, AMGPN, FEAT and UNEM are selected by registry identity. Deterministic episode indices, a fixed test root, matching backbone shapes and matching preprocessing are checked. Original training budgets/checkpoint provenance must still be supplied and audited externally. RFUAV evaluation uses fixed learned parameters and performs no training.


In [ ]:
from pathlib import Path
import json
from math import sqrt
from statistics import mean, stdev
import torch
from torch.utils.data import DataLoader, Subset
from amgpn.config import require as cfg, PROJECT_ROOT
from amgpn.data.episodes import UAVDataset, MetaDataset
from amgpn.experiments import create_experiment, group

method_ids = group("paper-main")["members"]
checkpoint_paths = cfg("workflow.evaluation.checkpoints")
model_kwargs = cfg("workflow.evaluation.model_kwargs")
trainer_kwargs = cfg("workflow.evaluation.trainer_kwargs")
assert set(method_ids) <= set(checkpoint_paths), "Provide a checkpoint for each method, including PN."
test_data = UAVDataset(cfg("workflow.evaluation.data_root"), seed=cfg("workflow.dataset_seed"),
                       is_train=True, train_ratio=1.0, max_sample_count=cfg("workflow.max_sample_count"))
n_way, n_trials = cfg("workflow.n_way"), cfg("workflow.evaluation.n_trials")
assert n_trials > 0
results = []
reference_shapes = reference_preprocessing = None
for method_id in method_ids:
    components = create_experiment(method_id, device=cfg("workflow.device"),
                                   model_kwargs=model_kwargs.get(method_id, {}),
                                   trainer_kwargs=trainer_kwargs.get(method_id, {}))
    trainer = components.trainer
    trainer.load_model(checkpoint_paths[method_id])
    for module in vars(trainer).values():
        if isinstance(module, torch.nn.Module):
            module.eval()
    shapes = {key: tuple(value.shape) for key, value in components.model.state_dict().items()}
    preprocessing = (trainer.crop_ratio_h, trainer.crop_ratio_l, trainer.resize)
    if reference_shapes is None:
        reference_shapes, reference_preprocessing = shapes, preprocessing
    assert shapes == reference_shapes, "Main comparison requires matching backbone architecture."
    assert preprocessing == reference_preprocessing, "Main comparison requires matching preprocessing."
    for k_shot in cfg("workflow.evaluation.k_shots"):
        episodes = MetaDataset(test_data, n_way, k_shot, cfg("workflow.q_query"),
                               num_tasks=n_trials, seed=cfg("workflow.episode_seed"))
        scores = []
        for trial in range(n_trials):
            loader = DataLoader(Subset(episodes, [trial]), batch_size=1, shuffle=False)
            scores.append(float(trainer.evaluate(loader, n_way, show_progress=False, show_error_stats=False)))
        dispersion = stdev(scores) if n_trials > 1 else 0.0
        results.append({"experiment": method_id, "k_shot": k_shot, "accuracy_mean": mean(scores),
                        "accuracy_std": dispersion, "accuracy_ci95": 1.96 * dispersion / sqrt(n_trials),
                        "episodes": n_trials})
    del trainer, components
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
output = Path(cfg("workflow.output_dir")).expanduser().resolve()
assert not output.is_relative_to(PROJECT_ROOT), "Keep experiment results outside the repository."
output.mkdir(parents=True, exist_ok=True)
(output / "paper_main_accuracy.json").write_text(json.dumps(results, indent=2))
results